Use the *vocal_pipeline_2025* environment in 3.9.0

# Imports

In [2]:
#load imports
import pandas as pd
import numpy as np
import librosa
import soundfile as sf
import ast
from pathlib import Path
from tqdm import tqdm
from skimage.transform import resize
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim
import matplotlib.pyplot as plt
import shutil
from PIL import Image
import logging
import dash
from dash import dcc, html, Input, Output
import plotly.graph_objects as go
from sklearn.neighbors import KDTree
import base64
import os
import sys
module_path = os.path.abspath("LocalMAP")
sys.path.insert(0, module_path)
from localmap import LocalMAP
from sklearn.cluster import Birch
import csv

/Users/masonyoungblood/anaconda3/envs/vocal_pipeline_2025/lib/python3.9/site-packages/numba/cpython/hashing.py:482: UserWarning: FNV hashing is not implemented in Numba. See PEP 456 https://www.python.org/dev/peps/pep-0456/ for rationale over not using FNV. Numba will continue to work, but hashes for built in types will be computed using siphash24. This will permit e.g. dictionaries to continue to behave as expected, however anything relying on the value of the hash opposed to hash as a derived property is likely to not work as expected.
  warnings.warn(msg)
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


# Config and data prep

### Spectrogram and model config

In [ ]:
#spectrogram parameters
SR = 44100
N_FFT = 2048
WIN_LENGTH = 1024
HOP_LENGTH = 128
N_MELS = 224
FMIN = 500
FMAX = 12000

#target shape for the vae input
TARGET_MELS = 128
TARGET_FRAMES = 128
TARGET_SHAPE = (TARGET_MELS, TARGET_FRAMES)

#vae training parameters
LATENT_DIM = 32
EPOCHS = 100
BATCH_SIZE = 128
LR = 1e-4
BETA = 0.5 #beta < 1.0 prioritizes reconstruction quality

#path configuration
CSV_FILE_PATH = Path("TweetyNoLongCleaned.csv")
ORIGINAL_RECORDINGS_DIR = Path("catbirds")
PROCESSED_DATA_DIR = Path("processed_recordings")
FEATURES_SAVE_PATH = Path("latent_features.npy")

### Preprocessing functions

In [ ]:
#normalizes a spectrogram to the [0, 1] range
def normalize_spec(spec):
    min_val = np.min(spec)
    max_val = np.max(spec)
    if max_val > min_val:
        return (spec - min_val) / (max_val - min_val)
    else:
        return np.zeros_like(spec)

#pads the time axis of a spectrogram to a target length, centering the content
def pad_center_zero(spec, target_frames):
    pad_total = target_frames - spec.shape[1]
    pad_left = pad_total // 2
    pad_right = pad_total - pad_left
    return np.pad(
        spec,
        ((0, 0), (pad_left, pad_right)),
        mode = 'constant',
        constant_values = 0
    )

#downsamples a spectrogram to a target shape using smooth interpolation
def downsample_spectrogram(spec, target_shape):
    return resize(spec, target_shape, anti_aliasing = True, preserve_range = True)

### Main preprocessing script

In [ ]:
print("--- Starting Data Preparation ---")
PROCESSED_DATA_DIR.mkdir(parents = True, exist_ok = True)

print(f"Loading data from: {CSV_FILE_PATH}")
try:
    df = pd.read_csv(CSV_FILE_PATH)
except FileNotFoundError:
    print(f"Error: CSV file not found at {CSV_FILE_PATH}. Please check the path.")
    sys.exit()

#convert string representations of lists to actual python lists
for col in ['onsets', 'offsets']:
    if col in df.columns:
        df[col] = df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
    else:
        print(f"Warning: Column '{col}' not found in CSV. This might lead to errors.")

#pre-calculate the maximum frame length needed for padding
print("Calculating maximum note duration for padding...")
max_frames = 0
for index, row in tqdm(df.iterrows(), total = len(df), desc = "Scanning Durations"):
    for onset, offset in zip(row['onsets'], row['offsets']):
        start_frame = librosa.time_to_frames(onset, sr = SR, hop_length = HOP_LENGTH)
        end_frame = librosa.time_to_frames(offset, sr = SR, hop_length = HOP_LENGTH)
        num_frames = end_frame - start_frame
        if num_frames > max_frames:
            max_frames = num_frames
print(f"Maximum duration found: {max_frames} frames. This will be the padding target.")

#process each recording, generate, process, and save spectrograms
print(f"Processing {len(df)} recordings...")
for index, row in tqdm(df.iterrows(), total = len(df), desc = "Processing Recordings"):
    filename = row['filename']
    onsets = row['onsets']
    offsets = row['offsets']

    original_wav_path = ORIGINAL_RECORDINGS_DIR / filename
    if not original_wav_path.exists():
        print(f"Warning: Original WAV file not found for {filename}. Skipping.")
        continue

    try:
        y_full, sr_loaded = librosa.load(original_wav_path, sr = SR)
    except Exception as e:
        print(f"Error loading {original_wav_path}: {e}. Skipping.")
        continue

    #compute mel spectrogram for the entire recording once for efficiency
    try:
        full_mel_spectrogram = librosa.feature.melspectrogram(
            y = y_full, sr = SR, n_fft = N_FFT, win_length = WIN_LENGTH,
            hop_length = HOP_LENGTH, n_mels = N_MELS, fmin = FMIN, fmax = FMAX
        )
        full_mel_spectrogram_db = librosa.power_to_db(full_mel_spectrogram, ref = np.max)
    except Exception as e:
        print(f"Error computing full spectrogram for {filename}: {e}. Skipping.")
        continue

    base_filename_no_ext = original_wav_path.stem

    #iterate through each note in the current recording
    for i, (onset, offset) in enumerate(zip(onsets, offsets)):
        try:
            #convert onset/offset times to spectrogram frame indices
            start_frame = librosa.time_to_frames(onset, sr = SR, hop_length = HOP_LENGTH)
            end_frame = librosa.time_to_frames(offset, sr = SR, hop_length = HOP_LENGTH)

            #ensure frames are within bounds
            start_frame = max(0, start_frame)
            end_frame = min(full_mel_spectrogram_db.shape[1], end_frame)

            if start_frame >= end_frame:
                continue

            #slice the full spectrogram
            note_spectrogram_db = full_mel_spectrogram_db[:, start_frame:end_frame]

            #process the spectrogram immediately ---
            normalized_spec = normalize_spec(note_spectrogram_db)
            padded_spec = pad_center_zero(normalized_spec, max_frames)
            final_spec = downsample_spectrogram(padded_spec, TARGET_SHAPE)

            #save the final, processed spectrogram slice as a .npy file
            output_npy_name = f"{base_filename_no_ext}_{i}.npy"
            output_npy_path = PROCESSED_DATA_DIR / output_npy_name
            np.save(output_npy_path, final_spec.astype(np.float32))

        except Exception as e:
            print(f"Error processing spectrogram segment {i} for {filename}: {e}.")

print("\n--- Data Preparation Complete! ---")
print(f"All processed spectrograms are saved in: {PROCESSED_DATA_DIR.resolve()}")

# Conv-VAE

### Define model structure

In [ ]:
class ConvEncoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 64, kernel_size = 4, stride = 2, padding = 1)
        self.bn1 = nn.BatchNorm2d(64)
        self.conv2 = nn.Conv2d(64, 128, kernel_size = 4, stride = 2, padding = 1)
        self.bn2 = nn.BatchNorm2d(128)
        self.conv3 = nn.Conv2d(128, 256, kernel_size = 4, stride = 2, padding = 1)
        self.bn3 = nn.BatchNorm2d(256)
        self.conv4 = nn.Conv2d(256, 512, kernel_size = 4, stride = 2, padding = 1)
        self.bn4 = nn.BatchNorm2d(512)
        self.flat_size = 512 * 8 * 8
        self.fc = nn.Linear(self.flat_size, 1024)
        self.fc_mu = nn.Linear(1024, latent_dim)
        self.fc_log_var = nn.Linear(1024, latent_dim)

    def forward(self, x):
        x = F.mish(self.bn1(self.conv1(x)))
        x = F.mish(self.bn2(self.conv2(x)))
        x = F.mish(self.bn3(self.conv3(x)))
        x = F.mish(self.bn4(self.conv4(x)))
        x = x.view(-1, self.flat_size)
        x = F.mish(self.fc(x))
        mu = self.fc_mu(x)
        log_var = self.fc_log_var(x)
        return mu, log_var

class ConvDecoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.flat_size = 512 * 8 * 8
        self.fc1 = nn.Linear(latent_dim, 1024)
        self.fc2 = nn.Linear(1024, self.flat_size)
        self.deconv1 = nn.ConvTranspose2d(512, 256, kernel_size = 4, stride = 2, padding = 1)
        self.bn1 = nn.BatchNorm2d(256)
        self.deconv2 = nn.ConvTranspose2d(256, 128, kernel_size = 4, stride = 2, padding = 1)
        self.bn2 = nn.BatchNorm2d(128)
        self.deconv3 = nn.ConvTranspose2d(128, 64, kernel_size = 4, stride = 2, padding = 1)
        self.bn3 = nn.BatchNorm2d(64)
        self.deconv4 = nn.ConvTranspose2d(64, 1, kernel_size = 4, stride = 2, padding = 1)

    def forward(self, z):
        x = F.mish(self.fc1(z))
        x = F.mish(self.fc2(x))
        x = x.view(-1, 512, 8, 8)
        x = F.mish(self.bn1(self.deconv1(x)))
        x = F.mish(self.bn2(self.deconv2(x)))
        x = F.mish(self.bn3(self.deconv3(x)))
        x_recon = torch.sigmoid(self.deconv4(x))
        return x_recon

class SimpleConvVAE(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.encoder = ConvEncoder(latent_dim)
        self.decoder = ConvDecoder(latent_dim)

    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, log_var = self.encoder(x)
        z = self.reparameterize(mu, log_var)
        x_recon = self.decoder(z)
        return x_recon, mu, log_var

def vae_loss(x, x_recon, mu, log_var, beta = 1.0):
    #reconstruction loss (l1 is often better for image-like data than bce)
    recon_loss = F.l1_loss(x_recon, x, reduction = 'sum')
    #kl divergence (analytical formula for efficiency)
    kld = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())
    #average over batch
    return (recon_loss + beta * kld) / x.size(0)

### Load data and model

In [ ]:
print("\n--- Starting VAE Training ---")
NPY_DIR = PROCESSED_DATA_DIR
npy_files = sorted(list(NPY_DIR.glob("*.npy")))
spectrograms = []
names = []

for npy_file in tqdm(npy_files, desc = "Loading Processed NPY files"):
    spec = np.load(npy_file)
    spectrograms.append(spec)
    names.append(npy_file.name)

#data is already processed, just convert to tensor
vae_data = torch.tensor(np.array(spectrograms), dtype = torch.float32).unsqueeze(1)
#clamp to avoid exact 0s and 1s
vae_data = torch.clamp(vae_data, 1e-6, 1 - 1e-6)

dataset = TensorDataset(vae_data)
dataloader = DataLoader(dataset, batch_size = BATCH_SIZE, shuffle = True)

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = SimpleConvVAE(latent_dim = LATENT_DIM).to(device)
optimizer = optim.AdamW(model.parameters(), lr = LR)

### Train model

In [ ]:
print("Starting training...")

#initialize a list to store average loss per epoch
loss_history = []

for epoch in range(EPOCHS):
    model.train()
    total_epoch_loss = 0
    progress_bar = tqdm(dataloader, desc = f"Epoch {epoch+1}/{EPOCHS}", leave = False)

    for batch in progress_bar:
        x_img = batch[0].to(device)

        optimizer.zero_grad()
        x_recon_img, mu, log_var = model(x_img)
        loss = vae_loss(x_img, x_recon_img, mu, log_var, beta = BETA)
        loss.backward()
        optimizer.step()

        total_epoch_loss += loss.item()
        progress_bar.set_postfix(loss = f"{loss.item():.4f}")

    avg_loss = total_epoch_loss / len(dataloader)
    #append the average loss for this epoch
    loss_history.append(avg_loss)
    print(f"Epoch {epoch+1}/{EPOCHS} | Average Loss: {avg_loss:.4f}")

print("--- Training complete. ---")

#save model and loss history
MODEL_SAVE_PATH = "catbird_vae_model.pth"
print(f"Saving model state to {MODEL_SAVE_PATH}...")
torch.save(model.state_dict(), MODEL_SAVE_PATH)

#save loss history
LOSS_HISTORY_PATH = "loss_history.npy"
print(f"\nSaving loss history to {LOSS_HISTORY_PATH}...")
np.save(LOSS_HISTORY_PATH, np.array(loss_history))

In [ ]:
#plot the loss curve
plt.figure(figsize=(10, 5))
plt.plot(loss_history, label="Training Loss")
plt.title("Training Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Average Loss")
plt.legend()
plt.grid(True)
plt.show()

# Evaluation and feature extraction

### Evaluation functions

In [ ]:
#plots a side-by-side comparison of original and reconstructed spectrograms
def plot_reconstructions(model, dataloader, device, num_examples = 8):
    model.eval()
    with torch.no_grad():
        x_batch, = next(iter(dataloader))
        x_img = x_batch[:num_examples].to(device)
        x_recon_img, _, _ = model(x_img)

    originals = x_img.cpu().squeeze(1)
    reconstructions = x_recon_img.cpu().squeeze(1)

    fig, axes = plt.subplots(2, num_examples, figsize = (16, 4))
    fig.suptitle('Originals vs. Reconstructions', fontsize = 16)

    for i in range(num_examples):
        axes[0, i].imshow(originals[i], cmap = 'viridis', origin = 'lower')
        axes[0, i].set_xticks([])
        axes[0, i].set_yticks([])
        axes[1, i].imshow(reconstructions[i], cmap = 'viridis', origin = 'lower')
        axes[1, i].set_xticks([])
        axes[1, i].set_yticks([])
        if i == 0:
            axes[0, i].set_ylabel("Original", fontsize = 12)
            axes[1, i].set_ylabel("Recon", fontsize = 12)

    plt.tight_layout(rect = [0, 0, 1, 0.95])
    plt.show()

def extract_latent_features(model, dataloader, device):
    #runs data through the model to extract latent features (mu)
    model.eval()
    all_features = []
    with torch.no_grad():
        progress_bar = tqdm(dataloader, desc = "Extracting Features")
        for batch in progress_bar:
            x = batch[0].to(device)
            #bug fix: simpleconvvae returns 3 values, not 6
            _, mu, _ = model(x)
            all_features.append(mu.cpu())

    latent_features_np = torch.cat(all_features, dim = 0).numpy()
    return latent_features_np

### Plot examples

In [ ]:
print("\nPlotting example reconstructions...")
plot_reconstructions(model, dataloader, device)

### Extract and save latent features

In [ ]:
infer_dataloader = DataLoader(dataset, batch_size = BATCH_SIZE, shuffle = False)
latent_features = extract_latent_features(model, infer_dataloader, device)

print(f"Extraction complete. Shape of latent features: {latent_features.shape}")
np.save(FEATURES_SAVE_PATH, latent_features)
print(f"Latent features saved to {FEATURES_SAVE_PATH}")